# Built-in BasicTS Model Benchmark

This section benchmarks several built-in BasicTS forecasting models on the ETTh1 dataset using the same shared settings and training procedure, then ranks them by performance.


In [5]:
# Project setup: ensure repo root and src are on the path
import os
import sys
from pathlib import Path
ROOT = Path(r"C:\Users\luwil\OneDrive\Documents\Code\BasicTS")
os.chdir(ROOT)
src_path = ROOT / "src"
if str(src_path) not in sys.path:
	sys.path.insert(0, str(src_path))


In [6]:
# Imports and shared benchmark settings
import json
import importlib
import traceback
import pkgutil
from pathlib import Path
from math import sqrt
import pandas as pd

import torch

from basicts.configs import BasicTSForecastingConfig, BasicTSModelConfig
from basicts.launcher import BasicTSLauncher
from basicts.runners.builder import Builder
from basicts.runners.taskflow import BasicTSForecastingTaskFlow
from basicts.scaler import ZScoreScaler
from basicts.utils import BasicTSMode

# Shared dataset / training settings for the benchmark
DATASET_NAME = "ETTh1"
INPUT_LEN = 96
OUTPUT_LEN = 12
NUM_FEATURES = 7
BATCH_SIZE = 32
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3

# Shared BasicTS config fields used for every model
SHARED_CONFIG = {
	"dataset_name": DATASET_NAME,
	"input_len": INPUT_LEN,
	"dataset_params": {
		"input_len": INPUT_LEN,
		"output_len": OUTPUT_LEN,
		"use_timestamps": False,
		"memmap": False,
	},
	"use_timestamps": False,
	"batch_size": BATCH_SIZE,
	"num_epochs": NUM_EPOCHS,
	"scaler": ZScoreScaler,
	"norm_each_channel": True,
	"rescale": False,
	"metrics": ["MAE", "MSE", "RMSE", "MAPE", "WAPE"],
	"optimizer_params": {"lr": LEARNING_RATE, "weight_decay": 5e-4},
	"gpus": None,
	"train_data_num_workers": 0,
	"val_data_num_workers": 0,
	"test_data_num_workers": 0,
	"save_results": True,
}

# Auto-discover model modules inside basicts.models that expose a Config and a forecasting class
import basicts.models as _models_pkg

def discover_models():
	candidates = []
	for finder, name, ispkg in pkgutil.iter_modules(_models_pkg.__path__):
		try:
			mod = importlib.import_module(f"basicts.models.{name}")
		except Exception:
			continue
		has_config = any(attr.endswith("Config") for attr in dir(mod))
		has_forecast = any("Forecast" in attr for attr in dir(mod))
		has_modelname = hasattr(mod, name)
		if has_config and (has_forecast or has_modelname):
			candidates.append(name)
	return sorted(candidates)

discovered = discover_models()
print("Discovered model modules:", discovered)

# All available models to test
ALL_MODELS = ["Autoformer", "Crossformer", "DLinear", "DUET", "FiLM", "FITS", "FreTS", "HI", "Informer", 
              "iTransformer", "Koopa", "Leddam", "LightTS", "MTSMixer", "NLinear", "NonstationaryTransformer", 
              "PatchTST", "SegRNN", "SOFTS", "SparseTSF", "StemGNN", "STID", "TiDE", "TimeKAN", "TimeMixer", 
              "Timer", "TimesNet", "TimeXer"]

# Build MODEL_REGISTRY by taking available models from ALL_MODELS list in order
MODEL_REGISTRY = []
for name in ALL_MODELS:
	if name in discovered:
		MODEL_REGISTRY.append(name)

print(f"Using MODEL_REGISTRY ({len(MODEL_REGISTRY)} models):", MODEL_REGISTRY)

# Where to save benchmark results
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

Discovered model modules: ['Autoformer', 'DLinear', 'FITS', 'FiLM', 'FreTS', 'HI', 'Informer', 'Koopa', 'Leddam', 'LightTS', 'MTSMixer', 'NLinear', 'NonstationaryTransformer', 'PatchTST', 'SOFTS', 'STID', 'SegRNN', 'SparseTSF', 'StemGNN', 'TiDE', 'TimeKAN', 'TimeMixer', 'TimeXer', 'Timer', 'TimesNet', 'iTransformer']
Using MODEL_REGISTRY (26 models): ['Autoformer', 'DLinear', 'FiLM', 'FITS', 'FreTS', 'HI', 'Informer', 'iTransformer', 'Koopa', 'Leddam', 'LightTS', 'MTSMixer', 'NLinear', 'NonstationaryTransformer', 'PatchTST', 'SegRNN', 'SOFTS', 'SparseTSF', 'StemGNN', 'STID', 'TiDE', 'TimeKAN', 'TimeMixer', 'Timer', 'TimesNet', 'TimeXer']


In [ ]:
# Benchmark runner: imports model classes/configs, trains and evaluates, and collects metrics.
import os
from pathlib import Path
from glob import glob

def find_attr_case(module, candidates):
	for name in candidates:
		if hasattr(module, name):
			return getattr(module, name)
	return None

available_models = []
failed_models = []
results = []

for model_name in MODEL_REGISTRY:
	try:
		mod = importlib.import_module(f"basicts.models.{model_name}")
	except Exception as e:
		failed_models.append({"model": model_name, "error": f"import error: {e}"})
		continue

	# Heuristics to find a forecasting wrapper or a model class
	# Prefer classes mentioning 'Forecast' in their name, else fallback to class named like model_name
	model_cls = None
	for attr in dir(mod):
		if "Forecast" in attr:
			model_cls = getattr(mod, attr)
			break
	if model_cls is None and hasattr(mod, model_name):
		model_cls = getattr(mod, model_name)

	# Find a config dataclass
	config_cls = None
	if hasattr(mod, f"{model_name}Config"):
		config_cls = getattr(mod, f"{model_name}Config")
	else:
		# fallback: any attribute that endswith Config
		for attr in dir(mod):
			if attr.endswith("Config"):
				config_cls = getattr(mod, attr)
				break

	if model_cls is None or config_cls is None:
		failed_models.append({"model": model_name, "error": "missing model class or config in module"})
		continue

	print(f"Benchmarking {model_name}...")

	# instantiate model config with shared required fields; configs have defaults for optional args
	try:
		# Some config classes accept num_features, some don't; pass what is common
		model_config = config_cls(input_len=INPUT_LEN, output_len=OUTPUT_LEN, num_features=NUM_FEATURES)
	except TypeError:
		# fallback without num_features
		model_config = config_cls(input_len=INPUT_LEN, output_len=OUTPUT_LEN)

	ckpt_dir = Path(f"checkpoints/benchmark/{model_name}/{DATASET_NAME}_{INPUT_LEN}_{OUTPUT_LEN}")

	cfg = BasicTSForecastingConfig(
		model=model_cls,
		model_config=model_config,
		taskflow=BasicTSForecastingTaskFlow(),
		ckpt_save_dir=str(ckpt_dir),
		**SHARED_CONFIG,
	)

	# Train and evaluate, catching any errors so the loop continues
	try:
		BasicTSLauncher.launch_training(cfg)
	except Exception as e:
		tb = traceback.format_exc()
		failed_models.append({"model": model_name, "error": f"train error: {e}\n{tb}"})
		continue

	try:
		# Passing None lets the runner pick the best checkpoint according to target metric
		BasicTSLauncher.launch_evaluation(cfg, None)
	except Exception as e:
		tb = traceback.format_exc()
		failed_models.append({"model": model_name, "error": f"eval error: {e}\n{tb}"})
		# still try to find any saved metrics

	# Find the most recent test_metrics.json under the ckpt_dir tree
	metrics_files = list(Path(ckpt_dir).rglob("test_metrics.json"))
	if not metrics_files:
		# no metrics saved
		failed_models.append({"model": model_name, "error": "no test_metrics.json found after eval"})
		continue

	metrics_file = max(metrics_files, key=lambda p: p.stat().st_mtime)
	try:
		with open(metrics_file, "r") as f:
			metrics = json.load(f)
	except Exception as e:
		failed_models.append({"model": model_name, "error": f"failed reading metrics: {e}"})
		continue

	overall = metrics.get("overall", {})

	# Normalize / compute missing metrics
	mae = overall.get("MAE")
	mse = overall.get("MSE")
	rmse = overall.get("RMSE") if overall.get("RMSE") is not None else (sqrt(mse) if mse is not None else None)
	mape = overall.get("MAPE")
	wape = overall.get("WAPE")

	results.append({
		"model": model_name,
		"MAE": mae,
		"MSE": mse,
		"RMSE": rmse,
		"MAPE": mape,
		"WAPE": wape,
		"metrics_file": str(metrics_file),
		"ckpt_dir": str(ckpt_dir),
	})

# Save results
df = pd.DataFrame(results)
if not df.empty:
	df_sorted = df.sort_values(by=["MAE", "MSE"], na_position="last")
	df_sorted.to_csv(RESULTS_DIR / "model_benchmark_results.csv", index=False)
	df_sorted.to_json(RESULTS_DIR / "model_benchmark_results.json", orient="records", indent=2)

# Save failed models
fm_df = pd.DataFrame(failed_models)
if not fm_df.empty:
	fm_df.to_csv(RESULTS_DIR / "model_benchmark_failed.csv", index=False)

print("Benchmark finished. Results saved to results/")


2026-06-25 16:09:28,680 - BasicTS-launcher - INFO - Launching BasicTS training.
2026-06-25 16:09:28,687 - BasicTS - INFO - Building model.
2026-06-25 16:09:28,690 - BasicTS-launcher - INFO - Launching BasicTS training.
2026-06-25 16:09:28,696 - BasicTS - INFO - Building model.
2026-06-25 16:09:28,697 - BasicTS - INFO - Set ckpt save dir: "checkpoints\benchmark\DLinear\ETTh1_96_12\1e0a36e17a575e4f938c2d8d175d2888"
2026-06-25 16:09:28,698 - BasicTS-training - INFO - Initializing training.
2026-06-25 16:09:28,699 - BasicTS-training - INFO - Building train data loader.
2026-06-25 16:09:28,701 - BasicTS-training - INFO - Loading Checkpoint from 'checkpoints\benchmark\DLinear\ETTh1_96_12\1e0a36e17a575e4f938c2d8d175d2888\DLinear_5.pt'
2026-06-25 16:09:28,703 - BasicTS-training - INFO - Resume training
2026-06-25 16:09:28,708 - BasicTS-training - INFO - Set optim: Adam
2026-06-25 16:09:28,709 - BasicTS-training - INFO - Building val data loader.
2026-06-25 16:09:28,710 - BasicTS-training - INF

Benchmarking Autoformer...
Benchmarking DLinear...


100%|██████████| 87/87 [00:00<00:00, 336.60it/s]
2026-06-25 16:09:28,978 - BasicTS-training - INFO - Result <test>: [test/time: 0.26 (s), test/loss: 0.3313, test/MAE: 0.3313, test/MSE: 0.2827, test/RMSE: 0.5317, test/MAPE: 8.3728, test/WAPE: 0.5573]
2026-06-25 16:09:28,978 - BasicTS-training - INFO - Test results saved to checkpoints\benchmark\DLinear\ETTh1_96_12\1e0a36e17a575e4f938c2d8d175d2888\test_results.
2026-06-25 16:09:28,980 - BasicTS-training - INFO - Test metrics saved to checkpoints\benchmark\DLinear\ETTh1_96_12\1e0a36e17a575e4f938c2d8d175d2888\test_metrics.json.
2026-06-25 16:09:28,982 - BasicTS-launcher - INFO - Launching BasicTS evaluation.
2026-06-25 16:09:28,984 - BasicTS - INFO - Building model.
2026-06-25 16:09:28,985 - BasicTS - INFO - Set ckpt save dir: "checkpoints\benchmark\DLinear\ETTh1_96_12\1e0a36e17a575e4f938c2d8d175d2888"
2026-06-25 16:09:28,987 - BasicTS-evaluation - INFO - Loading Checkpoint from 'checkpoints\benchmark\DLinear\ETTh1_96_12\1e0a36e17a575e4f93

Benchmarking FiLM...


100%|██████████| 267/267 [01:36<00:00,  2.76it/s]
2026-06-25 16:11:06,214 - BasicTS-training - INFO - Result <train>: [train/time: 96.88 (s), train/loss: 0.3593, train/MAE: 0.3593, train/MSE: 0.2905, train/RMSE: 0.5390, train/MAPE: 4.7964, train/WAPE: 0.6780]
2026-06-25 16:11:06,218 - BasicTS-training - INFO - Start validation.
100%|██████████| 87/87 [00:11<00:00,  7.35it/s]
2026-06-25 16:11:18,054 - BasicTS-training - INFO - Result <val>: [val/time: 11.84 (s), val/loss: 0.3987, val/MAE: 0.3987, val/MSE: 0.3708, val/RMSE: 0.6089, val/MAPE: 5.5851, val/WAPE: 0.6480]
2026-06-25 16:11:18,090 - BasicTS-training - INFO - Checkpoint checkpoints\benchmark\FiLM\ETTh1_96_12\bb8de176c215edb0ab2eba999e74c0bf\FiLM_best_val_MAE.pt saved
100%|██████████| 87/87 [00:12<00:00,  7.11it/s]
2026-06-25 16:11:30,332 - BasicTS-training - INFO - Result <test>: [test/time: 12.24 (s), test/loss: 0.3410, test/MAE: 0.3410, test/MSE: 0.2984, test/RMSE: 0.5462, test/MAPE: 8.0388, test/WAPE: 0.5658]
2026-06-25 16:11

...

100%|██████████| 267/267 [00:09<00:00, 28.94it/s]
2026-06-25 17:07:40,728 - BasicTS-training - INFO - Result <train>: [train/time: 9.23 (s), train/loss: 0.3583, train/MAE: 0.3583, train/MSE: 0.2896, train/RMSE: 0.5382, train/MAPE: 4.8657, train/WAPE: 0.6691]
2026-06-25 17:07:40,731 - BasicTS-training - INFO - Start validation.
100%|██████████| 87/87 [00:01<00:00, 77.07it/s]
2026-06-25 17:07:41,862 - BasicTS-training - INFO - Result <val>: [val/time: 1.13 (s), val/loss: 0.3865, val/MAE: 0.3865, val/MSE: 0.3314, val/RMSE: 0.5756, val/MAPE: 5.8796, val/WAPE: 0.6142]
2026-06-25 17:07:41,875 - BasicTS-training - INFO - Checkpoint checkpoints\benchmark\SegRNN\ETTh1_96_12\7c2c61fec5a691ed12d768c3a2efc264\SegRNN_best_val_MAE.pt saved
100%|██████████| 87/87 [00:01<00:00, 79.95it/s]
2026-06-25 17:07:42,966 - BasicTS-training - INFO - Result <test>: [test/time: 1.09 (s), test/loss: 0.3540, test/MAE: 0.3540, test/MSE: 0.3079, test/RMSE: 0.5549, test/MAPE: 9.3250, test/WAPE: 0.5815]
2026-06-25 17:0

Benchmarking SOFTS...


  0%|          | 0/267 [00:00<?, ?it/s]
2026-06-25 17:08:30,765 - BasicTS-training - ERROR - Traceback (most recent call last):
  File "C:\Users\luwil\OneDrive\Documents\Code\BasicTS\src\basicts\launcher.py", line 109, in training_func
    runner.train()
  File "C:\Users\luwil\OneDrive\Documents\Code\BasicTS\src\basicts\runners\basicts_runner.py", line 306, in train
    self._train_loop()
  File "C:\Users\luwil\OneDrive\Documents\Code\BasicTS\src\basicts\runners\basicts_runner.py", line 451, in _train_loop
    forward_return = self._forward(self.model, data, self.global_steps, self.epoch)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\luwil\OneDrive\Documents\Code\BasicTS\src\basicts\runners\basicts_runner.py", line 787, in _forward
    forward_return = model(inputs, **kwargs)
                     ^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\luwil\miniconda3\envs\BasicTS\Lib\site-packages\torch\nn\modules\module.py", line 1739, in _wrap

Benchmarking SparseTSF...
Benchmarking StemGNN...


2026-06-25 17:08:30,972 - BasicTS - INFO - Set ckpt save dir: "checkpoints\benchmark\StemGNN\ETTh1_96_12\5dd4f66ff36862923825bb390e0c1907"
2026-06-25 17:08:30,973 - BasicTS-training - INFO - Initializing training.
2026-06-25 17:08:30,973 - BasicTS-training - INFO - Building train data loader.
2026-06-25 17:08:30,978 - BasicTS-training - INFO - Set optim: Adam
2026-06-25 17:08:30,979 - BasicTS-training - INFO - Building val data loader.
2026-06-25 17:08:30,980 - BasicTS-training - INFO - Building test data loader.
2026-06-25 17:08:30,981 - BasicTS-training - INFO - Total parameters: 67401527
2026-06-25 17:08:30,981 - BasicTS-training - INFO - Trainable parameters: 67401527
2026-06-25 17:08:30,982 - BasicTS-training - INFO - Epoch 1 / 5
100%|██████████| 267/267 [05:57<00:00,  1.34s/it]  
2026-06-25 17:14:28,885 - BasicTS-training - INFO - Result <train>: [train/time: 357.90 (s), train/loss: 0.5563, train/MAE: 0.5563, train/MSE: 0.5960, train/RMSE: 0.7720, train/MAPE: 6.0882, train/WAPE: 

In [ ]:
# Display final ranking and failures
import pandas as pd
results_csv = Path("results/model_benchmark_results.csv")
if results_csv.exists():
	df = pd.read_csv(results_csv)
	display(df.sort_values(["MAE", "MSE"], na_position="last"))
else:
	print("No successful model results found.")

failed_csv = Path("results/model_benchmark_failed.csv")
if failed_csv.exists():
	failed_df = pd.read_csv(failed_csv)
	print("\nFailed models:")
	display(failed_df)
else:
	print("No failures recorded.")


NameError: name 'Path' is not defined

: 